# Deep Learning ABSA Model Workflow

This notebook demonstrates how the deep learning model in this project works at prediction time.

The model performs **Aspect-Based Sentiment Analysis (ABSA)** for restaurant reviews. For one review, it predicts sentiment separately for each aspect:

- Food
- Service
- Price
- Eating Environment / Ambiance

For each aspect, the model predicts one of three labels:

- Positive
- Negative
- Unknown

`Unknown` means the review does not clearly discuss that aspect.

## 1. Import Libraries and Set Paths

The saved transformer model is expected to be in `bert_absa_model/` inside this folder.

In [15]:
from pathlib import Path
import json
import sys

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

cwd = Path.cwd()
candidate_dirs = [
    cwd,
    cwd / "backends" / "deep_learning",
]
DEEP_LEARNING_DIR = next(
    (path for path in candidate_dirs if (path / "predict_bert.py").exists()),
    cwd,
)
MODEL_DIR = DEEP_LEARNING_DIR / "bert_absa_model"
REPORT_PATH = DEEP_LEARNING_DIR / "bert_training_report.json"

if str(DEEP_LEARNING_DIR) not in sys.path:
    sys.path.insert(0, str(DEEP_LEARNING_DIR))

print("Deep learning folder:", DEEP_LEARNING_DIR)
print("Model folder:", MODEL_DIR)
print("Model exists:", MODEL_DIR.exists())

Deep learning folder: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/deep_learning
Model folder: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/deep_learning/bert_absa_model
Model exists: True


## 2. Define the Aspects and Labels

The model is trained as a 3-class classifier. It receives a review together with one aspect, then chooses `Positive`, `Negative`, or `Unknown`.

In [16]:
ASPECTS = [
    "Food",
    "Service",
    "Price",
    "Eating Environment / Ambiance",
]

ID2LABEL = {
    0: "Positive",
    1: "Negative",
    2: "Unknown",
}

pd.DataFrame({"label_id": list(ID2LABEL), "label": list(ID2LABEL.values())})

,label_id,label
0,0,Positive
1,1,Negative
2,2,Unknown


## 3. Start With One Review

At the API/app level, the user provides one plain text restaurant review.

In [17]:
review = "The food was delicious but the service was slow and the price was too high."
review

'The food was delicious but the service was slow and the price was too high.'

## 4. Convert One Review Into Four Model Inputs

The model does not classify the review only once. It creates one input per aspect.

So this single review becomes four `(review, aspect)` pairs.

In [18]:
aspect_rows = pd.DataFrame({
    "review": [review] * len(ASPECTS),
    "aspect": ASPECTS,
})

aspect_rows

,review,aspect
0,The food was delicious but the service was slo...,Food
1,The food was delicious but the service was slo...,Service
2,The food was delicious but the service was slo...,Price
3,The food was delicious but the service was slo...,Eating Environment / Ambiance


## 5. Load the Tokenizer and Model

The saved model is loaded from disk. The project calls this the BERT model, but the training script uses `microsoft/deberta-v3-base` by default, which is a BERT-like transformer model.

In [19]:
if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Missing local model folder: {MODEL_DIR}. "
        "Train the model or copy bert_absa_model into backends/deep_learning/."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, local_files_only=True)
model.to(device)
model.eval()

print("Device:", device)
print("Model type:", model.config.model_type)
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Device: cpu
Model type: deberta-v2
Number of labels: 3
id2label: {0: 'Positive', 1: 'Negative', 2: 'Unknown'}


## 6. Tokenize One `(Review, Aspect)` Pair

The tokenizer converts text into numeric token IDs. The model receives numbers, not raw words.

Here is what happens for the `Food` aspect only.

In [20]:
single_aspect = "Food"

single_encoded = tokenizer(
    review,
    single_aspect,
    truncation=True,
    max_length=160,
    return_tensors="pt",
)

tokens = tokenizer.convert_ids_to_tokens(single_encoded["input_ids"][0])

print("Aspect:", single_aspect)
print("Input IDs shape:", tuple(single_encoded["input_ids"].shape))
print("First tokens:")
tokens[:40]

Aspect: Food
Input IDs shape: (1, 20)
First tokens:


['[CLS]',
 '▁The',
 '▁food',
 '▁was',
 '▁delicious',
 '▁but',
 '▁the',
 '▁service',
 '▁was',
 '▁slow',
 '▁and',
 '▁the',
 '▁price',
 '▁was',
 '▁too',
 '▁high',
 '.',
 '[SEP]',
 '▁Food',
 '[SEP]']

## 7. Tokenize All Four Aspect Inputs as a Batch

During prediction, the code sends all four aspect rows through the model together as a batch.

In [21]:
encoded = tokenizer(
    [review] * len(ASPECTS),
    ASPECTS,
    truncation=True,
    padding=True,
    max_length=160,
    return_tensors="pt",
)

for key, value in encoded.items():
    print(key, tuple(value.shape))

input_ids (4, 24)
token_type_ids (4, 24)
attention_mask (4, 24)


## 8. Run the Model

The model outputs logits. Logits are raw class scores.

Then `softmax` converts those scores into probabilities for the three labels.

In [22]:
encoded_on_device = {key: value.to(device) for key, value in encoded.items()}

with torch.no_grad():
    outputs = model(**encoded_on_device)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1).cpu()
    predicted_ids = probabilities.argmax(dim=-1).tolist()

print("Logits shape:", tuple(logits.shape))
print("Probabilities shape:", tuple(probabilities.shape))

Logits shape: (4, 3)
Probabilities shape: (4, 3)


## 9. Inspect Probabilities and Final Predictions

Each row below is one aspect. The model chooses the label with the highest probability.

In [23]:
prediction_rows = []

for aspect, label_id, probs in zip(ASPECTS, predicted_ids, probabilities):
    prediction_rows.append({
        "aspect": aspect,
        "positive_probability": round(float(probs[0]), 4),
        "negative_probability": round(float(probs[1]), 4),
        "unknown_probability": round(float(probs[2]), 4),
        "predicted_sentiment": ID2LABEL[int(label_id)],
        "confidence": round(float(probs[label_id]), 4),
    })

pd.DataFrame(prediction_rows)

,aspect,positive_probability,negative_probability,unknown_probability,predicted_sentiment,confidence
0,Food,0.9368,0.0104,0.0527,Positive,0.9368
1,Service,0.0305,0.8998,0.0697,Negative,0.8998
2,Price,0.0349,0.8904,0.0746,Negative,0.8904
3,Eating Environment / Ambiance,0.0007,0.0003,0.9990,Unknown,0.9990


## 10. Convert Results Into the API Output Format

This is the same shape used by the backend API and Flutter app.

In [24]:
api_output = {
    "review": review,
    "method": "bert",
    "results": [
        {
            "aspect": row["aspect"],
            "sentiment": row["predicted_sentiment"],
            "confidence": row["confidence"],
        }
        for row in prediction_rows
    ],
}

print(json.dumps(api_output, indent=2, ensure_ascii=False))

{
  "review": "The food was delicious but the service was slow and the price was too high.",
  "method": "bert",
  "results": [
    {
      "aspect": "Food",
      "sentiment": "Positive",
      "confidence": 0.9368
    },
    {
      "aspect": "Service",
      "sentiment": "Negative",
      "confidence": 0.8998
    },
    {
      "aspect": "Price",
      "sentiment": "Negative",
      "confidence": 0.8904
    },
    {
      "aspect": "Eating Environment / Ambiance",
      "sentiment": "Unknown",
      "confidence": 0.999
    }
  ]
}


## 11. Compare With the Project Predictor Class

The project already wraps the same steps inside `BertAbsaPredictor` in `predict_bert.py`.

In [25]:
from predict_bert import BertAbsaPredictor

predictor = BertAbsaPredictor(model_dir=MODEL_DIR)
wrapped_output = predictor.predict(review)

print(json.dumps(wrapped_output, indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{
  "review": "The food was delicious but the service was slow and the price was too high.",
  "method": "bert",
  "results": [
    {
      "aspect": "Food",
      "sentiment": "Positive",
      "confidence": 0.9368
    },
    {
      "aspect": "Service",
      "sentiment": "Negative",
      "confidence": 0.8998
    },
    {
      "aspect": "Price",
      "sentiment": "Negative",
      "confidence": 0.8904
    },
    {
      "aspect": "Eating Environment / Ambiance",
      "sentiment": "Unknown",
      "confidence": 0.999
    }
  ]
}


## 12. Training Summary

The training report stores the model name, dataset size, and evaluation metrics from the saved training run.

In [26]:
if REPORT_PATH.exists():
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
    print("Model:", report.get("model"))
    print("Rows:", report.get("rows"))
    print("Test metrics:")
    for metric, value in report.get("test_metrics", {}).items():
        print(f"- {metric}: {value:.4f}")
else:
    print("No training report found at", REPORT_PATH)

Model: microsoft/deberta-v3-base
Rows: {'train': 9728, 'validation': 1216, 'test': 1220}
Test metrics:
- accuracy: 0.9197
- macro_precision: 0.8352
- macro_recall: 0.8185
- macro_f1: 0.8205
- weighted_precision: 0.9250
- weighted_recall: 0.9197
- weighted_f1: 0.9206


## 13. Try a Review

Change the text below and rerun the cell.

In [27]:
your_review = "The restaurant looked beautiful, but the food was cold and expensive."

print(json.dumps(predictor.predict(your_review), indent=2, ensure_ascii=False))

{
  "review": "The restaurant looked beautiful, but the food was cold and expensive.",
  "method": "bert",
  "results": [
    {
      "aspect": "Food",
      "sentiment": "Negative",
      "confidence": 0.8878
    },
    {
      "aspect": "Service",
      "sentiment": "Unknown",
      "confidence": 0.9988
    },
    {
      "aspect": "Price",
      "sentiment": "Negative",
      "confidence": 0.926
    },
    {
      "aspect": "Eating Environment / Ambiance",
      "sentiment": "Unknown",
      "confidence": 0.9925
    }
  ]
}
